In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [20]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [21]:
v1.dot(dv)

np.float32(0.32332397)

In [22]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [23]:
v2.dot(dv)

np.float32(0.019730438)

In [24]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 12:19:47--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py.7’

ingest.py.7         100%[===================>]     888  --.-KB/s    in 0s      

2026-06-23 12:19:47 (43.8 MB/s) - ‘ingest.py.7’ saved [888/888]



In [25]:
from ingest import load_faq_data

documents = load_faq_data()

In [26]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [27]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [28]:
import numpy as np
X = np.array(vectors)

In [29]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [30]:
scores = X.dot(v_query)

In [31]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [32]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [33]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [34]:
scores = np.array(scores)

top5 = np.argsort(scores)[-5:][::-1]

top5_scores = scores[top5]

top5, top5_scores

(array([  2, 625, 907, 538,   7]),
 array([0.76294106, 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
       dtype=float32))

In [35]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'

In [36]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [37]:
vindex.search(v1)

[{'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  'doc_id': '3f1424af17'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.",
  'doc_id': '2d8b16c2a0'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'quest

In [38]:
vindex.search(v1, num_results = 5, filter_dict = {"course": "llm-zoom"})
vindex

In [39]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [40]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [41]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [42]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [43]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [44]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [45]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join.\n\nYou don’t need a confirmation email or prior registration to start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [46]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [47]:
vs_index.fit(vectors, documents)

In [48]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [50]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [51]:
vs_index.close()